Direct download pattern:

https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{YEAR}_{MONTH}.zip

e.g.:

bash
wget --no-check-certificate https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip

Each zip is one month, contains all carriers and all airports for that month already — no per-carrier/per-airport looping needed at all, that's only relevant if you use the interactive web UI at Departures.aspx, which is the wrong entry point for bulk downloads. Just loop YEAR (1987–present) × MONTH (1–12), which is a clean, trivial Airflow DAG:

python
import requests

BASE = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"

for year in range(2018, 2026):
    for month in range(1, 13):
        url = BASE.format(year=year, month=month)
        # download, unzip, load into GCS/staging

Each zip contains one CSV (the schema is the same as what you'd get from the interactive download builder — flight date, carrier, origin, dest, scheduled/actual times, delay-cause minutes, cancellations).

Worth noting for your DAG: BTS updates with roughly a 1–2 month lag, so don't request the most recent 1-2 months blindly — check what's actually published first, or your DAG will just get 404s for months that don't exist yet.

If you also want the DB1B ticket/itinerary survey (10% sample of actual tickets — origin/destination/fare, closer to what you worked with at Kiwi.com) rather than just delay stats, that's a separate PREZIP pattern:

https://transtats.bts.gov/PREZIP/Origin_and_Destination_Survey_DB1BCoupon_{year}_{quarter}.zip

quarterly, not monthly, 1993–present.

In [4]:
!wget --no-check-certificate -O data/tmp.zip https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip

--2026-08-10 19:43:07--  https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
Resolving transtats.bts.gov (transtats.bts.gov)... 204.68.194.70
Connecting to transtats.bts.gov (transtats.bts.gov)|204.68.194.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27573265 (26M) [application/x-zip-compressed]
Saving to: ‘data/tmp.zip’

data/tmp.zip        100%[===================>]  26.30M   305KB/s    in 1m 50s  

2026-08-10 19:44:58 (245 KB/s) - ‘data/tmp.zip’ saved [27573265/27573265]



In [6]:
!unzip -o data/tmp.zip -d ./data/extracted/

Archive:  data/tmp.zip
  inflating: ./data/extracted//On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv  
  inflating: ./data/extracted//readme.html  


In [9]:
import pandas as pd

In [10]:
df = pd.read_csv('data/extracted/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv')

/var/folders/05/3xzngw_n6tv7wlp__kmhfjwh0000gn/T/ipykernel_52592/2765050087.py:1: DtypeWarning: Columns (0: Div2Airport, 1: Div2TailNum, 2: Div3Airport, 3: Div3TailNum) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/extracted/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv')


In [12]:
df.describe()

,Year,Quarter,Month,DayofMonth,DayOfWeek,DOT_ID_Reporting_Airline,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
count,547271.0,547271.0,547271.0,547271.000000,547271.000000,547271.000000,547271.000000,547271.000000,5.472710e+05,547271.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,2024.0,1.0,1.0,15.893364,3.802931,19943.650341,2344.884295,12659.022795,1.265906e+06,31751.846465,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.0,0.0,0.0,8.954236,2.012839,373.913358,1576.254272,1526.276446,1.526274e+05,1320.563507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2024.0,1.0,1.0,1.000000,1.000000,19393.000000,1.000000,10135.000000,1.013506e+06,30070.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2024.0,1.0,1.0,8.000000,2.000000,19790.000000,1083.000000,11292.000000,1.129202e+06,30647.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2024.0,1.0,1.0,16.000000,4.000000,19805.000000,2069.000000,12889.000000,1.288904e+06,31454.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2024.0,1.0,1.0,24.000000,6.000000,20363.000000,3454.000000,14027.000000,1.402702e+06,32467.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2024.0,1.0,1.0,31.000000,7.000000,20452.000000,8819.000000,16869.000000,1.686902e+06,35991.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
for column in df.columns: print(column)

Year
Quarter
Month
DayofMonth
DayOfWeek
FlightDate
Reporting_Airline
DOT_ID_Reporting_Airline
IATA_CODE_Reporting_Airline
Tail_Number
Flight_Number_Reporting_Airline
OriginAirportID
OriginAirportSeqID
OriginCityMarketID
Origin
OriginCityName
OriginState
OriginStateFips
OriginStateName
OriginWac
DestAirportID
DestAirportSeqID
DestCityMarketID
Dest
DestCityName
DestState
DestStateFips
DestStateName
DestWac
CRSDepTime
DepTime
DepDelay
DepDelayMinutes
DepDel15
DepartureDelayGroups
DepTimeBlk
TaxiOut
WheelsOff
WheelsOn
TaxiIn
CRSArrTime
ArrTime
ArrDelay
ArrDelayMinutes
ArrDel15
ArrivalDelayGroups
ArrTimeBlk
Cancelled
CancellationCode
Diverted
CRSElapsedTime
ActualElapsedTime
AirTime
Flights
Distance
DistanceGroup
CarrierDelay
WeatherDelay
NASDelay
SecurityDelay
LateAircraftDelay
FirstDepTime
TotalAddGTime
LongestAddGTime
DivAirportLandings
DivReachedDest
DivActualElapsedTime
DivArrDelay
DivDistance
Div1Airport
Div1AirportID
Div1AirportSeqID
Div1WheelsOn
Div1TotalGTime
Div1LongestGTime
Div1W

In [14]:
len(df.columns)

110

In [ ]:
df.isnull()

)

AttributeError: 'function' object has no attribute 'count'